In [0]:
%sql
use catalog sathya_databricks123;

create schema if not exists employee_attendance;

use schema employee_attendance;

In [0]:
%sql
select current_catalog();

select current_schema();

current_schema()
employee_attendance


In [0]:
attendance_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/sathya_databricks123/default/filestore/attendance.csv")

tasks_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/sathya_databricks123/default/filestore/tasks.csv")

employee_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/sathya_databricks123/default/filestore/employee.csv")

In [0]:
display(attendance_df)

display(tasks_df)

display(employee_df)

attendance_id,employee_id,attendance_date,clock_in,clock_out
1,101,2026-06-01,2026-06-01T09:00:00.000Z,2026-06-01T18:00:00.000Z
2,102,2026-06-01,2026-06-01T09:20:00.000Z,2026-06-01T17:45:00.000Z
3,103,2026-06-01,2026-06-01T09:05:00.000Z,2026-06-01T18:15:00.000Z
4,104,2026-06-01,2026-06-01T10:15:00.000Z,2026-06-01T17:00:00.000Z
5,105,2026-06-01,2026-06-01T08:50:00.000Z,2026-06-01T17:30:00.000Z
6,106,2026-06-01,2026-06-01T09:10:00.000Z,2026-06-01T18:05:00.000Z
7,107,2026-06-01,2026-06-01T09:00:00.000Z,2026-06-01T18:20:00.000Z
8,108,2026-06-01,2026-06-01T09:30:00.000Z,2026-06-01T17:40:00.000Z
9,109,2026-06-01,2026-06-01T08:45:00.000Z,2026-06-01T18:10:00.000Z
10,110,2026-06-01,2026-06-01T09:50:00.000Z,2026-06-01T17:15:00.000Z


task_id,employee_id,task_name,tasks_completed,task_status
1,101,dashboard development,8,completed
2,102,recruitment drive,5,completed
3,103,budget analysis,10,completed
4,104,social media campaign,3,pending
5,105,sales reporting,7,completed
6,106,software testing,8,completed
7,107,audit review,9,completed
8,108,candidate screening,5,completed
9,109,sales target review,10,completed
10,110,marketing presentation,4,pending


employee_id,employee_name,department,designation,joining_date
101,arun,it,developer,2023-01-10
102,priya,hr,executive,2022-06-15
103,karthik,finance,analyst,2021-08-20
104,sana,marketing,associate,2024-02-01
105,rahul,sales,executive,2023-07-12
106,meena,it,tester,2023-05-18
107,vikram,finance,accountant,2022-09-22
108,divya,hr,recruiter,2024-01-05
109,ajay,sales,manager,2021-11-11
110,nisha,marketing,executive,2023-03-15


In [0]:
attendance_df.printSchema()

tasks_df.printSchema()

employee_df.printSchema()

root
 |-- attendance_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- attendance_date: date (nullable = true)
 |-- clock_in: timestamp (nullable = true)
 |-- clock_out: timestamp (nullable = true)

root
 |-- task_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- task_name: string (nullable = true)
 |-- tasks_completed: integer (nullable = true)
 |-- task_status: string (nullable = true)

root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- designation: string (nullable = true)
 |-- joining_date: date (nullable = true)



In [0]:
attendance_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.employee_attendance.bronze_attendance"
)

In [0]:
tasks_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.employee_attendance.bronze_tasks"
)

In [0]:
employee_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.employee_attendance.bronze_employee"
)

In [0]:
%sql
select * from bronze_attendance;

select * from bronze_tasks;

select * from bronze_employee;

employee_id,employee_name,department,designation,joining_date
101,arun,it,developer,2023-01-10
102,priya,hr,executive,2022-06-15
103,karthik,finance,analyst,2021-08-20
104,sana,marketing,associate,2024-02-01
105,rahul,sales,executive,2023-07-12
106,meena,it,tester,2023-05-18
107,vikram,finance,accountant,2022-09-22
108,divya,hr,recruiter,2024-01-05
109,ajay,sales,manager,2021-11-11
110,nisha,marketing,executive,2023-03-15


In [0]:
attendance = spark.table(
"sathya_databricks123.employee_attendance.bronze_attendance"
)

tasks = spark.table(
"sathya_databricks123.employee_attendance.bronze_tasks"
)

employee = spark.table(
"sathya_databricks123.employee_attendance.bronze_employee"
)

In [0]:
attendance = attendance.dropDuplicates()

tasks = tasks.dropDuplicates()

employee = employee.dropDuplicates()

In [0]:
silver_df = attendance.join(
tasks,
"employee_id",
"inner"
).join(
employee,
"employee_id",
"inner"
)

display(silver_df)

employee_id,attendance_id,attendance_date,clock_in,clock_out,task_id,task_name,tasks_completed,task_status,employee_name,department,designation,joining_date
102,12,2026-06-02,2026-06-02T09:25:00.000Z,2026-06-02T17:30:00.000Z,12,employee onboarding,4,completed,priya,hr,executive,2022-06-15
104,14,2026-06-02,2026-06-02T10:30:00.000Z,2026-06-02T16:45:00.000Z,4,social media campaign,3,pending,sana,marketing,associate,2024-02-01
104,4,2026-06-01,2026-06-01T10:15:00.000Z,2026-06-01T17:00:00.000Z,4,social media campaign,3,pending,sana,marketing,associate,2024-02-01
109,9,2026-06-01,2026-06-01T08:45:00.000Z,2026-06-01T18:10:00.000Z,9,sales target review,10,completed,ajay,sales,manager,2021-11-11
106,16,2026-06-02,2026-06-02T09:15:00.000Z,2026-06-02T18:00:00.000Z,16,bug fixing,9,completed,meena,it,tester,2023-05-18
103,3,2026-06-01,2026-06-01T09:05:00.000Z,2026-06-01T18:15:00.000Z,13,financial planning,9,completed,karthik,finance,analyst,2021-08-20
102,2,2026-06-01,2026-06-01T09:20:00.000Z,2026-06-01T17:45:00.000Z,12,employee onboarding,4,completed,priya,hr,executive,2022-06-15
108,8,2026-06-01,2026-06-01T09:30:00.000Z,2026-06-01T17:40:00.000Z,18,training coordination,5,completed,divya,hr,recruiter,2024-01-05
101,1,2026-06-01,2026-06-01T09:00:00.000Z,2026-06-01T18:00:00.000Z,11,attendance report,7,completed,arun,it,developer,2023-01-10
107,17,2026-06-02,2026-06-02T09:05:00.000Z,2026-06-02T18:15:00.000Z,7,audit review,9,completed,vikram,finance,accountant,2022-09-22


In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
"work_hours",
(
col("clock_out").cast("timestamp").cast("long") -
col("clock_in").cast("timestamp").cast("long")
)/3600
)

In [0]:
silver_df = silver_df.withColumn(
"effective_work_hours",
col("work_hours")-1
)

In [0]:
silver_df = silver_df.withColumn(
"productivity_score",
col("tasks_completed")/
col("effective_work_hours")
)

display(silver_df)

employee_id,attendance_id,attendance_date,clock_in,clock_out,task_id,task_name,tasks_completed,task_status,employee_name,department,designation,joining_date,work_hours,effective_work_hours,productivity_score
102,12,2026-06-02,2026-06-02T09:25:00.000Z,2026-06-02T17:30:00.000Z,12,employee onboarding,4,completed,priya,hr,executive,2022-06-15,8.083333333333334,7.083333333333334,0.5647058823529412
104,14,2026-06-02,2026-06-02T10:30:00.000Z,2026-06-02T16:45:00.000Z,4,social media campaign,3,pending,sana,marketing,associate,2024-02-01,6.25,5.25,0.5714285714285714
104,4,2026-06-01,2026-06-01T10:15:00.000Z,2026-06-01T17:00:00.000Z,4,social media campaign,3,pending,sana,marketing,associate,2024-02-01,6.75,5.75,0.5217391304347826
109,9,2026-06-01,2026-06-01T08:45:00.000Z,2026-06-01T18:10:00.000Z,9,sales target review,10,completed,ajay,sales,manager,2021-11-11,9.416666666666666,8.416666666666666,1.1881188118811883
106,16,2026-06-02,2026-06-02T09:15:00.000Z,2026-06-02T18:00:00.000Z,16,bug fixing,9,completed,meena,it,tester,2023-05-18,8.75,7.75,1.1612903225806452
103,3,2026-06-01,2026-06-01T09:05:00.000Z,2026-06-01T18:15:00.000Z,13,financial planning,9,completed,karthik,finance,analyst,2021-08-20,9.166666666666666,8.166666666666666,1.1020408163265307
102,2,2026-06-01,2026-06-01T09:20:00.000Z,2026-06-01T17:45:00.000Z,12,employee onboarding,4,completed,priya,hr,executive,2022-06-15,8.416666666666666,7.416666666666666,0.5393258426966293
108,8,2026-06-01,2026-06-01T09:30:00.000Z,2026-06-01T17:40:00.000Z,18,training coordination,5,completed,divya,hr,recruiter,2024-01-05,8.166666666666666,7.166666666666666,0.6976744186046512
101,1,2026-06-01,2026-06-01T09:00:00.000Z,2026-06-01T18:00:00.000Z,11,attendance report,7,completed,arun,it,developer,2023-01-10,9.0,8.0,0.875
107,17,2026-06-02,2026-06-02T09:05:00.000Z,2026-06-02T18:15:00.000Z,7,audit review,9,completed,vikram,finance,accountant,2022-09-22,9.166666666666666,8.166666666666666,1.1020408163265307


In [0]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.employee_attendance.silver_employee"
)

In [0]:
%sql
select * from silver_employee;

employee_id,attendance_id,attendance_date,clock_in,clock_out,task_id,task_name,tasks_completed,task_status,employee_name,department,designation,joining_date,work_hours,effective_work_hours,productivity_score
102,12,2026-06-02,2026-06-02T09:25:00.000Z,2026-06-02T17:30:00.000Z,12,employee onboarding,4,completed,priya,hr,executive,2022-06-15,8.083333333333334,7.083333333333334,0.5647058823529412
104,14,2026-06-02,2026-06-02T10:30:00.000Z,2026-06-02T16:45:00.000Z,4,social media campaign,3,pending,sana,marketing,associate,2024-02-01,6.25,5.25,0.5714285714285714
104,4,2026-06-01,2026-06-01T10:15:00.000Z,2026-06-01T17:00:00.000Z,4,social media campaign,3,pending,sana,marketing,associate,2024-02-01,6.75,5.75,0.5217391304347826
109,9,2026-06-01,2026-06-01T08:45:00.000Z,2026-06-01T18:10:00.000Z,9,sales target review,10,completed,ajay,sales,manager,2021-11-11,9.416666666666666,8.416666666666666,1.1881188118811883
106,16,2026-06-02,2026-06-02T09:15:00.000Z,2026-06-02T18:00:00.000Z,16,bug fixing,9,completed,meena,it,tester,2023-05-18,8.75,7.75,1.1612903225806452
103,3,2026-06-01,2026-06-01T09:05:00.000Z,2026-06-01T18:15:00.000Z,13,financial planning,9,completed,karthik,finance,analyst,2021-08-20,9.166666666666666,8.166666666666666,1.1020408163265307
102,2,2026-06-01,2026-06-01T09:20:00.000Z,2026-06-01T17:45:00.000Z,12,employee onboarding,4,completed,priya,hr,executive,2022-06-15,8.416666666666666,7.416666666666666,0.5393258426966293
108,8,2026-06-01,2026-06-01T09:30:00.000Z,2026-06-01T17:40:00.000Z,18,training coordination,5,completed,divya,hr,recruiter,2024-01-05,8.166666666666666,7.166666666666666,0.6976744186046512
101,1,2026-06-01,2026-06-01T09:00:00.000Z,2026-06-01T18:00:00.000Z,11,attendance report,7,completed,arun,it,developer,2023-01-10,9.0,8.0,0.875
107,17,2026-06-02,2026-06-02T09:05:00.000Z,2026-06-02T18:15:00.000Z,7,audit review,9,completed,vikram,finance,accountant,2022-09-22,9.166666666666666,8.166666666666666,1.1020408163265307


In [0]:
from pyspark.sql.functions import avg

gold_employee = silver_df.groupBy(
"employee_id",
"employee_name"
).agg(
avg("effective_work_hours").alias("average_work_hours"),
avg("productivity_score").alias("average_productivity")
)

display(gold_employee)

employee_id,employee_name,average_work_hours,average_productivity
102,priya,7.25,0.6210178453403834
104,sana,5.5,0.45548654244306414
109,ajay,8.333333333333332,1.2001200120012
106,meena,7.833333333333332,1.0852292020373513
103,karthik,8.166666666666666,1.1632653061224492
108,divya,6.916666666666667,0.7238372093023255
101,arun,7.958333333333332,0.9424342105263158
107,vikram,8.25,1.0304081632653062
105,rahul,7.708333333333332,0.9730014025245443
110,nisha,6.458333333333334,0.6193806193806194


In [0]:
gold_employee.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.employee_attendance.gold_employee_productivity"
)

In [0]:
gold_department = silver_df.groupBy(
"department"
).agg(
avg("effective_work_hours").alias("average_work_hours"),
avg("productivity_score").alias("average_productivity")
)

display(gold_department)

department,average_work_hours,average_productivity
hr,7.083333333333332,0.6724275273213545
marketing,5.8194444444444455,0.5101179014222493
sales,8.020833333333332,1.0865607072628722
it,7.895833333333332,1.0138317062818336
finance,8.208333333333332,1.0968367346938777


In [0]:
gold_department.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.employee_attendance.gold_department_summary"
)

In [0]:
absentees = silver_df.filter(
col("effective_work_hours") < 7
)

display(absentees)

employee_id,attendance_id,attendance_date,clock_in,clock_out,task_id,task_name,tasks_completed,task_status,employee_name,department,designation,joining_date,work_hours,effective_work_hours,productivity_score
110,20,2026-06-02,2026-06-02T10:00:00.000Z,2026-06-02T17:30:00.000Z,10,marketing presentation,4,pending,nisha,marketing,executive,2023-03-15,7.5,6.5,0.6153846153846154
108,18,2026-06-02,2026-06-02T09:40:00.000Z,2026-06-02T17:20:00.000Z,18,training coordination,5,completed,divya,hr,recruiter,2024-01-05,7.666666666666667,6.666666666666667,0.75
104,4,2026-06-01,2026-06-01T10:15:00.000Z,2026-06-01T17:00:00.000Z,4,social media campaign,3,pending,sana,marketing,associate,2024-02-01,6.75,5.75,0.5217391304347826
110,10,2026-06-01,2026-06-01T09:50:00.000Z,2026-06-01T17:15:00.000Z,10,marketing presentation,4,pending,nisha,marketing,executive,2023-03-15,7.416666666666667,6.416666666666667,0.6233766233766234
104,14,2026-06-02,2026-06-02T10:30:00.000Z,2026-06-02T16:45:00.000Z,4,social media campaign,3,pending,sana,marketing,associate,2024-02-01,6.25,5.25,0.5714285714285714
108,18,2026-06-02,2026-06-02T09:40:00.000Z,2026-06-02T17:20:00.000Z,8,candidate screening,5,completed,divya,hr,recruiter,2024-01-05,7.666666666666667,6.666666666666667,0.75
104,4,2026-06-01,2026-06-01T10:15:00.000Z,2026-06-01T17:00:00.000Z,14,content creation,2,completed,sana,marketing,associate,2024-02-01,6.75,5.75,0.34782608695652173
104,14,2026-06-02,2026-06-02T10:30:00.000Z,2026-06-02T16:45:00.000Z,14,content creation,2,completed,sana,marketing,associate,2024-02-01,6.25,5.25,0.38095238095238093


In [0]:
absentees.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.employee_attendance.gold_absentees"
)

In [0]:
from pyspark.sql.functions import sum

gold_task_summary = silver_df.groupBy(
"employee_id",
"employee_name"
).agg(
sum("tasks_completed").alias("total_tasks")
)

display(gold_task_summary)

employee_id,employee_name,total_tasks
103,karthik,38
101,arun,30
110,nisha,8
108,divya,20
102,priya,18
104,sana,10
106,meena,34
107,vikram,34
109,ajay,40
105,rahul,30


In [0]:
gold_task_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.employee_attendance.gold_task_summary"
)

In [0]:
%sql
select * from gold_employee_productivity;

select * from gold_department_summary;

select * from gold_absentees;

select * from gold_task_summary;

employee_id,employee_name,total_tasks
108,divya,20
101,arun,30
105,rahul,30
110,nisha,8
104,sana,10
109,ajay,40
103,karthik,38
107,vikram,34
102,priya,18
106,meena,34


In [0]:
%sql show tables

database,tableName,isTemporary
employee_attendance,bronze_attendance,false
employee_attendance,bronze_employee,false
employee_attendance,bronze_tasks,false
employee_attendance,gold_absentees,false
employee_attendance,gold_department_summary,false
employee_attendance,gold_employee_productivity,false
employee_attendance,gold_task_summary,false
employee_attendance,silver_employee,false


In [0]:
spark.table(
"sathya_databricks123.employee_attendance.bronze_attendance"
).toPandas().to_csv(
"bronze_attendance.csv",
index=False
)

spark.table(
"sathya_databricks123.employee_attendance.bronze_tasks"
).toPandas().to_csv(
"bronze_tasks.csv",
index=False
)

spark.table(
"sathya_databricks123.employee_attendance.bronze_employee"
).toPandas().to_csv(
"bronze_employee.csv",
index=False
)

In [0]:
spark.table(
"sathya_databricks123.employee_attendance.silver_employee"
).toPandas().to_csv(
"silver_employee.csv",
index=False
)

In [0]:
spark.table(
"sathya_databricks123.employee_attendance.gold_employee_productivity"
).toPandas().to_csv(
"gold_employee_productivity.csv",
index=False
)

spark.table(
"sathya_databricks123.employee_attendance.gold_department_summary"
).toPandas().to_csv(
"gold_department_summary.csv",
index=False
)

spark.table(
"sathya_databricks123.employee_attendance.gold_absentees"
).toPandas().to_csv(
"gold_absentees.csv",
index=False
)

spark.table(
"sathya_databricks123.employee_attendance.gold_task_summary"
).toPandas().to_csv(
"gold_task_summary.csv",
index=False
)